# PairRecon column mapping

This notebook reads one Pair from `pairs.xlsx`, queries the real Oracle and Databricks schemas through PairRecon's `src` functions, proposes deterministic name matches, and upserts only the rows you approve. Credentials remain in the Oracle environment variables and Databricks profile.

In [ ]:
from pathlib import Path

import pandas as pd

from pair_recon.app.config import env_value, load_environment
from pair_recon.app.runner import StatementSql, pair_config, workspace
from pair_recon.core.metadata import companion_table
from pair_recon.core.metadata.mapping import (
    automatch_columns,
    discover_databricks_columns,
    discover_oracle_columns,
    upsert_column_mappings,
)
from pair_recon.core.metadata.workbook import read_pair_workbook

load_environment()
PROFILE = env_value("DBX_PROFILE", default="corp")
WAREHOUSE_ID = env_value("DBX_WAREHOUSE_ID", default="")
PAIR_TABLE = env_value("DBX_PAIR_TABLE", default="catalog.schema.pair_recon_pair_meta")
PAIR_NAME = ""
WORKBOOK = Path(env_value("DBX_PAIR_WORKBOOK", default="setup/pairs.xlsx")).expanduser()

In [ ]:
rows = {row["pair_name"]: row for row in read_pair_workbook(WORKBOOK)}
if PAIR_NAME not in rows:
    raise ValueError(f"PAIR_NAME must be one of: {', '.join(sorted(rows))}")
row = rows[PAIR_NAME]
effective = pair_config(row)
client = workspace(PROFILE)
sql = StatementSql(client, WAREHOUSE_ID)
COLUMN_MAP_TABLE = companion_table(PAIR_TABLE, "pair_recon_column_map")

In [ ]:
left_columns = discover_oracle_columns(effective["left"])
right_columns = discover_databricks_columns(sql, effective["right"])
candidates = pd.DataFrame(automatch_columns(left_columns, right_columns))
display(candidates)

Review unmatched or ambiguous rows above. The next cell keeps only deterministic matches. Add, remove, or edit rows in `approved` and set an optional `data_type` (`numeric`, `categorical`, a registered date type, or custom JSON) before applying.

In [ ]:
approved = candidates.loc[
    candidates["status"] == "matched",
    ["left_column", "right_column", "data_type"],
].copy()
# Example manual correction:
# approved.loc[approved["left_column"] == "AMOUNT", "data_type"] = "numeric"
display(approved)

In [ ]:
APPLY = False
if APPLY:
    result = upsert_column_mappings(
        sql, COLUMN_MAP_TABLE, PAIR_NAME, approved.to_dict(orient="records")
    )
    display(result)
else:
    print("Preview only. Set APPLY = True after reviewing approved mappings.")